# ColabUMA TS — UMAによる遷移状態探索

Metaの機械学習ポテンシャル **UMA** で、反応の **遷移状態（TS）** を探すノートブックです。コードを書く必要はありません。

1. **DMF**（Direct MaxFlux 法、[PyDMF](https://github.com/shin1koda/dmf)）で反応物と生成物をつなぐ反応経路を最適化し、経路上のエネルギーの最高点を TS の初期構造にします
2. **Sella** で1次の鞍点（TS）に最適化します。最初のヘシアンは UMA の数値微分で作ります
3. 振動計算で虚振動が1本だけか確かめ、虚振動の向きに少しずらして両側に最適化して、反応物と生成物につながっているかを確かめます（簡易IRC）
4. **DFT の一点計算**（[GPU4PySCF](https://github.com/pyscf/gpu4pyscf)）で、UMA の構造のまま反応物・TS・生成物のエネルギーを計算し、活性化エネルギーを比べます

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に名前 `HF_TOKEN`、値にHugging Faceのアクセストークンを登録し、「ノートブックからのアクセス」をオンにする
3. 下の **1. 入力フォーム** で出力名、電荷、スピン多重度、TS の初期構造の作り方、DFT の条件を指定する
4. **ランタイム → すべて実行** を押す
5. **2. ファイル選択と事前確認** で、反応物と生成物のファイル（xyz または gjf）を順に選ぶ。「TSの初期構造（Sellaのみ）」なら TS の初期構造のファイルを1つ選ぶ。あとは放置できます
6. 下のセルで経路・TS・虚振動・エネルギーを確認し、最後のセルで結果のZIPをダウンロードする

**反応物と生成物は、同じ原子が同じ順番で並んだ構造にしてください**（DMF は原子の対応を番号で決めます）。スキャンの結果（`preopt.xyz` と `postopt.xyz`）はそのまま使えます。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## 出力（ZIPの中身）

| ファイル | 内容 |
|---|---|
| `<出力名>_ts.xyz` | Sella で最適化した TS |
| `<出力名>_ts_opt.log` | TS 最適化の経路（Gaussian の opt=ts の書式） |
| `<出力名>_ts_freq.log` | TS の振動数と基準振動（Gaussian の freq の書式。GaussView で虚振動のアニメーションが見られる） |
| `<出力名>_ts_freq.txt` | 振動数の一覧 |
| `<出力名>_ts.gjf` | Gaussian で TS を DFT で確かめるための入力（ルート行はフォームで指定） |
| `<出力名>_dmf_path.xyz` / `.log` / `.csv` | DMF で最適化した反応経路（等間隔の31点＋最高点）とエネルギー |
| `<出力名>_dmf_init.xyz` | FB-ENM で作った初期経路 |
| `<出力名>_ts_guess.xyz` | DMF の経路上の最高点（Sella の初期構造） |
| `<出力名>_reactant.xyz` / `product.xyz` | 反応物と生成物（端点を最適化したときはその結果。`_opt.log` も付く） |
| `<出力名>_irc_minus.xyz` / `irc_plus.xyz` | 虚振動の向きにずらして最適化した構造 |
| `<出力名>_dft.csv` | DFT 一点計算のエネルギー |
| `<出力名>_log.txt` | 入力条件、ソフトのバージョン、各段階の結果 |

## 注意事項

- 見つかる TS は、UMA のポテンシャル面での TS です。DFT の一点計算は UMA の構造のままなので、DFT での TS の位置とは少しずれます。最終的には ZIP の gjf などで DFT の TS 最適化と IRC を行ってください。
- 「簡易IRC」は IRC そのものではありません。虚振動の向きに一番動く原子を 0.15 Å ずらしてから最適化し、結合のつながりが反応物・生成物と同じかを見ています。
- 電荷とスピン多重度は全体で固定です。開殻の DFT は UKS で計算します。
- DFT は GPU4PySCF（密度フィッティング）で計算します。T4 は倍精度の計算が遅いため、原子数が多いときや def2-TZVPD では時間がかかります。GPU が使えないときは CPU の PySCF で計算します。
- 初回は DMF のための IPOPT のビルドなどで、インストールに数分かかります。
- UMAは機械学習ポテンシャルであり、DFTの代わりにはなりません。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabChem#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

遷移状態探索（DMF・Sella）を使った場合は、次も引用してください。

- S.-i. Koda and S. Saito, *J. Open Source Softw.* **11**, 10379 (2026).（PyDMF）
- S.-i. Koda and S. Saito, *J. Chem. Theory Comput.* **20**, 2798–2811 (2024).（Direct MaxFlux）
- S.-i. Koda and S. Saito, *J. Chem. Theory Comput.* **20**, 7176–7187 (2024); **21**, 3513–3522 (2025).（FB-ENM、CFB-ENM）
- E. D. Hermes, K. Sargsyan, H. N. Najm, J. Zádor, *J. Chem. Theory Comput.* **18**, 6974–6988 (2022).（Sella）
- DFT の一点計算を使った場合：PySCF（Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)）と GPU4PySCF（X. Wu *et al.*, arXiv:2404.09452 (2024)）

In [ ]:
#@title 1. 入力フォーム { display-mode: "form" }
#@markdown 入力が終わったら **ランタイム → すべて実行** を押してください。次のセルでファイルを選ぶと、あとは放置できます。

#@markdown ### 出力
output_name = "result"  #@param {type:"string"}

#@markdown ### 構造
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}
#@markdown 反応物と生成物から DMF で TS の初期構造を作るか、手元の TS の初期構造から Sella だけで最適化するかを選びます。
ts_input = "反応物と生成物（DMF → Sella）"  #@param ["反応物と生成物（DMF → Sella）", "TSの初期構造（Sellaのみ）"]
#@markdown DMF の前に、反応物と生成物をそれぞれ拘束なしで最適化します。
opt_endpoints = True  #@param {type:"boolean"}

#@markdown ### DMF（反応経路と TS の初期構造）
#@markdown 経路の途中でエネルギーを計算する点の数。多いほど TS の初期構造が正確になり、時間もかかります。
dmf_nmove = 5  #@param {type:"integer"}
dmf_convergence = "middle"  #@param ["loose", "middle", "tight"]
dmf_max_iter = 300  #@param {type:"integer"}

#@markdown ### Sella（TS の最適化）
#@markdown 最初のヘシアンを UMA の数値微分で作ります（力の計算 6N 回）。オフにすると近似ヘシアンで始めます。
ts_exact_hessian = True  #@param {type:"boolean"}
#@markdown TS から虚振動の向きに少しずらして両側に最適化し、どの構造につながるか確かめます（簡易IRC）。
check_connection = True  #@param {type:"boolean"}

#@markdown ### DFT 一点計算（gpu4pyscf）
#@markdown UMA の構造で DFT のエネルギーを計算し、活性化エネルギーを比べます。ωB97M-V/def2-TZVPD が UMA（OMol25）の学習データと同じ計算レベルです。def2-TZVPD は T4 では1点に数十分かかることがあります。
run_dft = True  #@param {type:"boolean"}
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V（OMol25と同じ）", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]
#@markdown ZIP に入れる Gaussian の TS 計算用 gjf のルート行。
gaussian_route = "# opt=(ts,calcfc,noeigentest) freq b3lyp/def2svp empiricaldispersion=gd3bj"  #@param {type:"string"}

#@markdown ### 計算条件
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown 収束判定 fmax（eV/Å）。normal は Gaussian の既定、tight は Gaussian の opt=tight と同じくらいです。
convergence = "normal（0.02）"  #@param ["loose（0.05）", "normal（0.02）", "tight（0.001）", "数値指定"]
#@markdown 「数値指定」を選んだときの fmax（eV/Å）。
fmax_custom = 0.02  #@param {type:"number"}
#@markdown 収束判定と最大ステップ数は、端点の最適化・TS の最適化・簡易IRC で共通です。
max_steps = 500  #@param {type:"integer"}

In [ ]:
#@title 2. ファイル選択と事前確認 { display-mode: "form" }
#@markdown ファイルの選択とトークンの確認をここで済ませます。このあとのインストールと計算は放置できます。

import os
import subprocess
import sys

COLABUMA_REF = "main"  # 共通コード（colabuma パッケージ）を取ってくる GitHub のブランチ


def _use_colabuma(ref):
    # ノートと同じフォルダに colabuma があればそれを、なければ GitHub から取ってきて使う
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"ColabChem の共通コードを GitHub から取得できませんでした（ブランチ {ref}）。\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, gaussian, opt, output, structure, ts, ui
from colabuma.ui import ColabUMAError, ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
use_dmf = ts_input.startswith("反応物")
if dmf_nmove < 1 or dmf_max_iter < 1:
    raise ColabUMAError("DMF の nmove と最大反復回数は1以上にしてください。", "通常は nmove = 5、最大反復回数 = 300 程度です。")
if run_dft:
    dft.parse_method(dft_functional, dft_basis)
env.check_gpu()
env.ensure_token()
if use_dmf:
    reactant_file = structure.upload("reactant", "反応物の xyz または gjf ファイル")
    product_file = structure.upload("product", "生成物の xyz または gjf ファイル")
    _n = [structure.quick_atom_count(*reactant_file), structure.quick_atom_count(*product_file)]
    if all(_n) and _n[0] != _n[1]:
        raise ColabUMAError(f"反応物（{_n[0]} 原子）と生成物（{_n[1]} 原子）の原子数が違います。",
                            "同じ分子の、原子の順番がそろった2つの構造を選んでください。")
else:
    guess_file = structure.upload("guess", "TS の初期構造の xyz または gjf ファイル")
ok("入力の確認が終わりました。ここからは放置できます。")

In [ ]:
#@title 3. セットアップ（インストール・認証） { display-mode: "form" }
DEVICE = env.setup(["ts", "dft"] if run_dft else ["ts"])

In [ ]:
#@title 4. モデル読み込み { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. 構造の準備 { display-mode: "form" }
if use_dmf:
    reactant0, _desc_r = structure.read_structure(*reactant_file, charge, multiplicity)
    product0, _desc_p = structure.read_structure(*product_file, charge, multiplicity)
    product0 = ts.check_endpoints(reactant0, product0)
    source_desc = f"反応物 {_desc_r} / 生成物 {_desc_p}"
    formed, broken = ts.bond_changes(reactant0, product0)
    print("反応物（番号付き。赤は結合が変わる原子）：")
    structure.show_numbered(reactant0, {i for b in formed + broken for i in b})
    print(f"できる結合: {ts.bond_text(formed, reactant0)}　切れる結合: {ts.bond_text(broken, reactant0)}")
    ok(f"{reactant0.get_chemical_formula()}（{len(reactant0)} 原子）の反応物と生成物を用意しました。")
else:
    guess0, source_desc = structure.read_structure(*guess_file, charge, multiplicity)
    structure.show_numbered(guess0)
    ok(f"{guess0.get_chemical_formula()}（{len(guess0)} 原子）の TS の初期構造を用意しました。")

In [ ]:
#@title 6. 反応経路（DMF） { display-mode: "form" }
#@markdown DMF の「収束」のバーは、IPOPT の双対残差が収束判定までに桁でどこまで近づいたかの目安です。
endpoints = dmf_res = None
reactant = product = None
if use_dmf:
    if opt_endpoints:
        print(f"反応物と生成物を {optimizer_name} で最適化します（fmax < {fmax} eV/Å）")
        endpoints = {"reactant": opt.optimize(reactant0, calc, job, "反応物"),
                     "product": opt.optimize(product0, calc, job, "生成物")}
        for _r in endpoints.values():
            opt.warn_unconverged(_r)
        reactant = endpoints["reactant"]["atoms"]
        product = ts.check_endpoints(reactant, endpoints["product"]["atoms"])
        endpoints["product"]["atoms"] = product
    else:
        reactant, product = reactant0, product0
    dmf_res = ts.run_dmf(reactant, product, calc, job, nmove=dmf_nmove, tol=dmf_convergence, max_iter=dmf_max_iter)
    if endpoints is None:
        endpoints = {"reactant": {"atoms": reactant, "energy": dmf_res["energies"][0]},
                     "product": {"atoms": product, "energy": dmf_res["energies"][-1]}}
    ts_guess = dmf_res["ts_guess"]
else:
    print("TS の初期構造が指定されたので、DMF は飛ばします。")
    ts_guess = guess0

In [ ]:
#@title 7. TS の最適化（Sella）と確認 { display-mode: "form" }
ts_res = ts.run_sella(ts_guess, calc, job, exact_hessian=ts_exact_hessian)
ts_vib = ts.check_ts(ts_res, calc)
conn = None
if check_connection and ts_vib is not None and ts_vib["n_imag"] >= 1:
    conn = ts.connect(ts_res["atoms"], ts_vib, calc, job, reactant, product)
files, ts_log, uma_rows = ts.report(job, reactant, product, dmf_res, ts_res, ts_vib, conn, endpoints)
files["ts.gjf"] = gaussian.gjf(ts_res["atoms"], job, gaussian_route, f"{job.out_prefix} TS from ColabUMA ({model_name})",
                                 chk=f"{job.out_prefix}_ts.chk")

In [ ]:
#@title 8. DFT 一点計算 { display-mode: "form" }
dft_log = []
if run_dft:
    dft_res = dft.run(ts.dft_points(ts_res, endpoints, conn), charge, multiplicity, dft_functional, dft_basis)
    _f, dft_log = dft.report(dft_res, uma_rows)
    files.update(_f)
else:
    print("DFT 一点計算はオフです。")

In [ ]:
#@title 9. 保存 { display-mode: "form" }
_cond = [f"TS の初期構造: {ts_input}", f"端点の最適化: {'オン' if (use_dmf and opt_endpoints) else 'オフ'}",
         f"Sella の最初のヘシアン: {'数値微分' if ts_exact_hessian else '近似'}", f"簡易IRC: {'オン' if check_connection else 'オフ'}"]
files["log.txt"] = output.log_text("ColabUMA TS 計算ログ", source_desc, job, [_cond, ts_log, dft_log])
zip_path = output.save_zip(job.out_prefix, files)